![image.png](https://i.imgur.com/a3uAqnb.png)

# Lab 1: Quantisation, Fewer Bits per Number

> **Note:** this is the worked solution. The student version, with the four tasks left for you, is `01_Quantisation_Exercise.ipynb`, one folder up.

A trained network stores every weight as a 32-bit float. **Quantisation** stores and computes with 8-bit integers instead: about 4x smaller, and faster on hardware with integer arithmetic, usually for a fraction of a point of accuracy. It is the first compression to try, because it needs no retraining. In this lab you will:

1. Train the Fashion-MNIST CNN from Day 2, about a minute on a laptop CPU.
2. Write int8 quantisation yourself, and see when **affine** beats **symmetric** (TASKS 1 and 2).
3. Quantise every weight from 8 bits down to 2, **per tensor** and **per channel** (TASK 3).
4. Get a real speed-up: run the CNN in int8 with **ONNX Runtime**.
5. Quantise a real object detector, **YOLO11**. Quantise everything and watch it break, then fix it (TASK 4).
6. On your own laptop: compare the YOLO versions live on your webcam.

It runs on a laptop CPU or on Colab; no GPU needed.

## Setup

On Colab, PyTorch is already installed; the next cell adds ONNX Runtime and Ultralytics (YOLO). On your own laptop, install the course's `requirements.txt` instead and skip that cell.

In [ ]:
# Colab only. On your laptop: pip install -r requirements.txt from the course repository.
%pip install -q onnx onnxruntime onnxslim ultralytics

In [ ]:
import copy
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Part 1: The Model to Compress

The CNN from Day 2's Fashion-MNIST lab: two convolutions, then two fully connected layers, 421,642 parameters. Two epochs of training get it to about 89% test accuracy in roughly a minute on a laptop CPU.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train_set = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=transform)
CLASSES = train_set.classes
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
X_test = torch.stack([image for image, _ in test_set])   # all 10,000 test images in one tensor
y_test = test_set.targets
print(f"{len(train_set):,} training images, {len(test_set):,} test images, {len(CLASSES)} classes")

In [ ]:
class SimpleFashionCNN(nn.Module):
    # Day 2's Fashion-MNIST CNN
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # [B, 32, 14, 14]
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # [B, 64, 7, 7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


@torch.no_grad()
def accuracy(model):
    model.eval()
    return (model(X_test).argmax(1) == y_test).float().mean().item()


def size_mb(model, bits=32):
    # storage for the weights at `bits` per number
    return sum(p.numel() for p in model.parameters()) * bits / 8 / 1e6


model = SimpleFashionCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
start = time.time()
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        F.cross_entropy(model(x), y).backward()
        optimizer.step()
    print(f"epoch {epoch + 1}: test accuracy {accuracy(model):.2%}")
print(f"trained in {time.time() - start:.0f} s; {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"{size_mb(model):.2f} MB as float32")

## Part 2: Quantisation by Hand

Quantisation maps each real number $x$ onto a grid of integers. The grid has a step $s$, the **scale**, and an integer offset $z$, the **zero point**:

$$q = \mathrm{clamp}\big(\mathrm{round}(x / s) + z,\; q_{\min},\; q_{\max}\big), \qquad \hat{x} = s\,(q - z)$$

For int8, $q_{\min} = -128$ and $q_{\max} = 127$. The gap between $x$ and $\hat{x}$ is the **rounding error**: at most half a step for values inside the grid's range, and larger for values outside it, which get clamped to the nearest end.

In [ ]:
def quantize(x, scale, zero_point, bits=8):
    # real numbers -> integers on the grid (kept in a float tensor, so we can look at them)
    q_min, q_max = -2 ** (bits - 1), 2 ** (bits - 1) - 1
    # TASK 1a: divide by the scale, round to the nearest integer, add the zero point, clamp to [q_min, q_max].
    return torch.clamp(torch.round(x / scale) + zero_point, q_min, q_max)


def dequantize(q, scale, zero_point):
    # integers -> the real numbers they stand for
    # TASK 1b: undo the mapping: x_hat = scale * (q - zero_point).
    return scale * (q - zero_point)


# A check with scale 0.1 and zero point 0: the grid covers -12.8 to 12.7 in steps of 0.1
x = torch.tensor([-13.0, -0.26, 0.0, 0.04, 1.234, 12.69, 50.0])
q = quantize(x, 0.1, 0)
print("x     ", x.tolist())
print("q     ", q.tolist())
print("x_hat ", [round(v, 3) for v in dequantize(q, 0.1, 0).tolist()])

Inside the range, every value moved by at most half a step (0.05). The two values outside it, -13 and 50, were clamped to the ends of the grid: one cost 0.2, the other 37.3. Choosing the range is the whole game.

### Symmetric or affine?

Two ways to fit the grid to a tensor:

- **Symmetric**: the zero point is 0 and the grid is centred on 0. The scale maps the largest magnitude onto $q_{\max}$: $s = \max|x| / q_{\max}$.
- **Affine** (asymmetric): the scale spreads the tensor's actual range $[x_{\min}, x_{\max}]$ over all the integers, $s = (x_{\max} - x_{\min}) / (q_{\max} - q_{\min})$, and the zero point maps $x_{\min}$ onto $q_{\min}$: $z = q_{\min} - \mathrm{round}(x_{\min} / s)$.

Both first stretch the range to include 0, so that 0 itself is exactly representable. Zeros are everywhere in a network: padding, ReLU outputs, and (in Lab 2) pruned weights.

In [ ]:
def choose_qparams(x, bits=8, symmetric=True, dim=None):
    # Scale and zero point for x. dim=None: one pair for the whole tensor ("per tensor");
    # dim=0: one pair for each output channel ("per channel").
    q_min, q_max = -2 ** (bits - 1), 2 ** (bits - 1) - 1
    if dim is None:
        lo, hi = x.min(), x.max()
    else:
        others = [d for d in range(x.dim()) if d != dim]
        lo, hi = x.amin(dim=others, keepdim=True), x.amax(dim=others, keepdim=True)
    lo, hi = torch.clamp(lo, max=0), torch.clamp(hi, min=0)      # stretch the range to include 0
    if symmetric:
        # TASK 2a: symmetric: the scale maps the largest magnitude, max(-lo, hi), onto q_max; the zero point is 0.
        scale = (torch.maximum(-lo, hi) / q_max).clamp(min=1e-12)
        zero_point = torch.zeros_like(scale)
    else:
        # TASK 2b: affine: the scale spreads [lo, hi] over (q_max - q_min) steps; the zero point maps lo onto q_min.
        scale = ((hi - lo) / (q_max - q_min)).clamp(min=1e-12)
        zero_point = q_min - torch.round(lo / scale)
    return scale, zero_point

Now compare the two on two very different tensors from the trained CNN: the second convolution's **weights**, and the **activations** coming out of the first ReLU.

In [ ]:
def round_trip_error(x, **kwargs):
    # mean |x - x_hat|, relative to mean |x|
    scale, zero_point = choose_qparams(x, **kwargs)
    x_hat = dequantize(quantize(x, scale, zero_point), scale, zero_point)
    return ((x_hat - x).abs().mean() / x.abs().mean()).item()


weights = model.features[3].weight.detach()            # conv2: 64 x 32 x 3 x 3
with torch.no_grad():
    model.eval()
    activations = model.features[:2](X_test[:256])    # after conv1 and its ReLU

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, (name, x) in zip(axes, [("conv2 weights", weights), ("ReLU activations", activations)]):
    ax.hist(x.flatten().numpy(), bins=100, log=True)
    ax.set_title(name)
    print(f"{name:17s} int8 rounding error: symmetric {round_trip_error(x, symmetric=True):.2%}, "
          f"affine {round_trip_error(x, symmetric=False):.2%}")
plt.tight_layout()
plt.show()

On the weights the two tie: weights are spread around 0, so a grid centred on 0 fits them. On the activations, affine roughly halves the error. ReLU outputs are never negative, so a symmetric grid keeps half of its 256 integers for negative numbers that never come, and its steps are twice as coarse. That is why toolkits quantise **weights symmetrically and activations affinely**, as ONNX Runtime does in Part 4.

## Part 3: Quantise the Whole Network: How Many Bits, and Per What?

Next, quantise every weight in the CNN. We **round-trip** them: quantise, then dequantise straight back, so PyTorch can still run the model. This "fake quantisation" is the standard way to measure the accuracy cost before deploying anything. Two choices matter:

- **Bits**: 8, or fewer.
- **Granularity**: one scale for each whole tensor, or one for each output channel. Per-channel scales cost one extra number per channel, a few hundred against 421,642 weights.

In [ ]:
def quantize_weights(model, bits, per_channel):
    # a copy of the model whose conv and linear weights went through symmetric quantisation and back
    q_model = copy.deepcopy(model)
    for module in q_model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            # TASK 3: a symmetric scale and zero point for w, one per output channel (dim=0) if per_channel,
            #         else one for the whole tensor (dim=None); then replace w by its round trip.
            scale, zero_point = choose_qparams(w, bits, symmetric=True, dim=0 if per_channel else None)
            module.weight.data = dequantize(quantize(w, scale, zero_point, bits), scale, zero_point)
    return q_model


float_acc = accuracy(model)
rows = []
for bits in (8, 6, 4, 3, 2):
    per_tensor = accuracy(quantize_weights(model, bits, per_channel=False))
    per_channel = accuracy(quantize_weights(model, bits, per_channel=True))
    rows.append({"bits": bits, "MB": size_mb(model, bits), "per tensor": per_tensor, "per channel": per_channel})
results = pd.DataFrame(rows).set_index("bits")
print(f"float32: {float_acc:.2%} at {size_mb(model):.2f} MB")
print(results.round(4).to_string())

plt.figure(figsize=(6, 3.5))
plt.plot(results.index, results["per tensor"], "o-", label="per tensor")
plt.plot(results.index, results["per channel"], "s-", label="per channel")
plt.axhline(float_acc, color="gray", ls="--", label="float32")
plt.gca().invert_xaxis()
plt.xlabel("bits per weight")
plt.ylabel("test accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Down to 4 bits the CNN barely notices. Below that, per-tensor scales fail first. Why? Look at how differently sized conv2's output channels are:

In [ ]:
ranges = weights.abs().amax(dim=(1, 2, 3))     # the largest |weight| in each of conv2's 64 output channels
plt.figure(figsize=(9, 2.5))
plt.bar(range(len(ranges)), ranges.numpy())
plt.xlabel("output channel of conv2")
plt.ylabel("max |weight|")
plt.title("A single scale for the whole tensor is set by the widest channel")
plt.show()
print(f"widest channel {ranges.max():.3f}, narrowest {ranges.min():.3f}: with one scale for the whole tensor, "
      f"the narrowest channel uses only about {ranges.min() / ranges.max():.0%} of the grid")

With one scale per tensor, a narrow channel falls between just a few grid steps, and at 2 or 3 bits it rounds to almost nothing. Per-channel scales give every channel the full grid. So: **8 bits is the safe default; below that, use per-channel scales**, and for 4 bits or fewer, usually quantisation-aware training, from the lecture.

## Part 4: A Real Speed-Up with ONNX Runtime

Fake quantisation measures the accuracy cost, but it still computes in float32: the model is no smaller and no faster. Real gains need a runtime with integer kernels.

- **ONNX** is a portable file format for trained networks.
- **ONNX Runtime** runs ONNX files on CPUs, phones and edge devices.

Its quantiser does what you just wrote, plus one step: **static** quantisation of the activations. It runs 200 calibration images through the network, records the range of every activation, and fixes those scales in the file, so nothing is measured at run time.

In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

model.eval()
torch.onnx.export(model, X_test[:1], "cnn_fp32.onnx", input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)


class CalibrationImages(CalibrationDataReader):
    # feeds n training images to the quantiser, one at a time, so it can record activation ranges
    def __init__(self, n=200):
        self.images = iter([{"input": train_set[i][0][None].numpy()} for i in range(n)])

    def get_next(self):
        return next(self.images, None)


quant_pre_process("cnn_fp32.onnx", "cnn_prepped.onnx")       # shape inference and graph clean-up
quantize_static("cnn_prepped.onnx", "cnn_int8.onnx", CalibrationImages(),
                quant_format=QuantFormat.QDQ, per_channel=True,              # per-channel weights, as in Part 3
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)  # affine activations, symmetric weights
print("wrote cnn_fp32.onnx and cnn_int8.onnx")

In [ ]:
def onnx_report(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1                 # one core: a steady, fair comparison
    session = ort.InferenceSession(path, options, providers=["CPUExecutionProvider"])
    X = X_test.numpy()
    preds = np.concatenate([session.run(None, {"input": X[i:i + 1000]})[0].argmax(1) for i in range(0, len(X), 1000)])

    def ms(batch, repeats):
        x = X[:batch]
        for _ in range(5):
            session.run(None, {"input": x})
        start = time.perf_counter()
        for _ in range(repeats):
            session.run(None, {"input": x})
        return (time.perf_counter() - start) / repeats * 1000

    return {"file": path, "MB": os.path.getsize(path) / 1e6, "accuracy": (preds == y_test.numpy()).mean(),
            "ms, 1 image": ms(1, 300), "ms, 256 images": ms(256, 20)}


report = pd.DataFrame([onnx_report("cnn_fp32.onnx"), onnx_report("cnn_int8.onnx")]).set_index("file")
print(report.round(4).to_string())
fp32, int8 = report.iloc[0], report.iloc[1]
print(f"\nint8: {fp32['MB'] / int8['MB']:.1f}x smaller, {fp32['ms, 256 images'] / int8['ms, 256 images']:.1f}x faster "
      f"on a batch, accuracy {100 * (int8['accuracy'] - fp32['accuracy']):+.2f} points")

About 4x smaller and faster, for a fraction of a point. How much faster depends on your CPU: integer kernels shine on chips with int8 dot-product instructions, which most laptops from the last few years have. That is the lecture's point about benchmarking on the target hardware: your speed-up is not your neighbour's.

## Part 5: A Real Detector, YOLO11

**YOLO11** (Ultralytics) finds and labels 80 kinds of objects in an image. We take its small version, **YOLO11s** (9.4 million parameters), export it to ONNX and quantise it with the same tool.

Accuracy for a detector is **mAP50-95**, the mean average precision over IoU thresholds from 0.5 to 0.95. Higher is better, and 0 means it finds nothing. We measure it on **COCO128**, 128 labelled photos. The first run downloads the model (19 MB) and COCO128 (7 MB).

In [ ]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import ASSETS
import cv2

coco = check_det_dataset("coco128.yaml")                       # downloads COCO128 the first time
yolo_fp32 = YOLO("yolo11s.pt").export(format="onnx", imgsz=640, verbose=False)
print("exported", yolo_fp32)


def letterbox(path, size=640):
    # YOLO's input: the image scaled to fit size x size, padded with grey, RGB, 0 to 1, shape [1, 3, size, size]
    image = cv2.imread(str(path))
    h, w = image.shape[:2]
    s = size / max(h, w)
    image = cv2.resize(image, (round(w * s), round(h * s)))
    canvas = np.full((size, size, 3), 114, np.uint8)
    canvas[:image.shape[0], :image.shape[1]] = image
    return (canvas[:, :, ::-1].transpose(2, 0, 1)[None] / 255.0).astype(np.float32)


class YoloCalibration(CalibrationDataReader):
    # 32 COCO photos for the activation ranges. (A real pipeline calibrates on data kept apart from the
    # evaluation set; COCO128 is all we downloaded, so these 32 are also among the 128 we evaluate on.)
    def __init__(self, n=32):
        self.images = iter([{"images": letterbox(f)} for f in sorted(Path(coco["train"]).glob("*.jpg"))[:n]])

    def get_next(self):
        return next(self.images, None)


def quantize_yolo(dst, nodes_to_exclude=()):
    quant_pre_process(yolo_fp32, "yolo_prepped.onnx")
    quantize_static("yolo_prepped.onnx", dst, YoloCalibration(), quant_format=QuantFormat.QDQ, per_channel=True,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                    nodes_to_exclude=list(nodes_to_exclude))
    original, quantised = onnx.load(yolo_fp32), onnx.load(dst)    # keep the class names YOLO stores in the file
    del quantised.metadata_props[:]
    quantised.metadata_props.extend(original.metadata_props)
    onnx.save(quantised, dst)
    return dst


def yolo_report(path):
    detector = YOLO(path, task="detect")
    for _ in range(3):
        detector.predict(ASSETS / "bus.jpg", verbose=False)       # warm up
    ms = np.median([sum(detector.predict(ASSETS / "bus.jpg", verbose=False)[0].speed.values()) for _ in range(20)])
    mAP = detector.val(data="coco128.yaml", imgsz=640, batch=1, plots=False, verbose=False).box.map
    return {"file": path, "MB": os.path.getsize(path) / 1e6, "ms per image": ms, "mAP50-95": mAP}


yolo_int8_all = quantize_yolo("yolo11s_int8_everything.onnx")
yolo_results = [yolo_report(yolo_fp32), yolo_report(yolo_int8_all)]
print(pd.DataFrame(yolo_results).set_index("file").round(3).to_string())

### Why it broke

The int8 model is faster and 4x smaller, but its mAP is 0: it finds nothing. The culprit is YOLO's last layer. It returns one tensor of shape (1, 84, 8400): for each of 8,400 candidate boxes, **4 box coordinates in pixels (0 to 640)** and **80 class scores (0 to 1)**. A single int8 scale has to cover 0 to 640, so its step is about 2.5, and every class score rounds to 0. Your own functions from Part 2 show it:

In [ ]:
box_and_scores = torch.tensor([612.0, 318.5, 41.2, 0.93, 0.41, 0.02])     # 3 coordinates in pixels, 3 class scores
scale, zero_point = choose_qparams(box_and_scores, symmetric=False)
print(f"scale {scale.item():.2f}")
print("after int8:", [round(v, 2) for v in dequantize(quantize(box_and_scores, scale, zero_point), scale, zero_point).tolist()])

The fix is to keep the head's **decoding** in float32 and quantise everything else. In the ONNX graph, YOLO11's detection head is the group of nodes whose names start with `/model.23/`. Its convolutions are ordinary layers, worth quantising. Everything after them (splitting, the softmax over box distances, the sigmoid scores, the final concatenation) should stay in float.

In [ ]:
graph = onnx.load("yolo_prepped.onnx").graph
# TASK 4: the names of the nodes to keep in float: every node of YOLO's head (name starts with "/model.23/")
#         except its convolutions (op_type "Conv").
head_nodes = [node.name for node in graph.node if node.name.startswith("/model.23/") and node.op_type != "Conv"]
print(f"keeping {len(head_nodes)} head nodes in float, for example {head_nodes[:3]}")

yolo_int8 = quantize_yolo("yolo11s_int8.onnx", nodes_to_exclude=head_nodes)
yolo_results.append(yolo_report(yolo_int8))
print(pd.DataFrame(yolo_results).set_index("file").round(3).to_string())

Now the int8 model keeps nearly all of its accuracy at a quarter of the size, and it is faster. The lesson goes beyond YOLO: **quantise layer by layer with your eyes open, and re-measure accuracy afterwards**. Any tensor that mixes very different ranges, like pixels and probabilities, needs its own treatment. That re-measurement is the lecture's release gate, and Lab 2 ends with one.

### See it live, on your own laptop

A notebook cannot show a live camera feed, and Colab cannot reach your webcam. On your own laptop, from the `Labs/Day_5/` folder of the course repository, run:

```
python yolo_live_demo.py
```

The first run builds four versions of **YOLO11m**, the medium model: big enough that the differences show. That takes a few minutes; after that it opens your camera. Keys 1 to 4 switch between the versions, and the corner of the window shows each one's frame time, FPS, size and mAP. `python yolo_live_demo.py --help` lists the options, such as `--camera 1` for another camera.

## Takeaways

- **Quantisation is a grid**: a scale and a zero point. Inside the grid's range the error is at most half a step; outside it, values are clamped.
- **Symmetric for weights, affine for activations.** ReLU outputs are never negative, so a symmetric grid wastes half its integers on them.
- **8 bits is nearly free; below that, per-channel scales matter**, and per-tensor scales collapse first.
- **Fake quantisation measures the cost; a runtime with integer kernels delivers the gain**: about 4x smaller, and faster on CPUs with int8 instructions.
- **Never quantise blindly.** One output tensor that mixed pixels with probabilities made YOLO useless; keeping its head in float fixed it. Always re-measure accuracy after compressing.